In [ ]:
import math

import plotly.graph_objects as go
import torch
import torch.nn.functional as F

from onomatron.data import Vocab, load_names, make_ngram_pairs, split_names
from onomatron.mlp import MLP
from onomatron.utils import lr_range_test, plot_history, plot_lr_range, train

# Load names, split train/val/test, build vocab

In [ ]:
names = load_names("../data/processed/names.txt")
train_names, val_names, test_names = split_names(names, seed=42)
vocab = Vocab(train_names)  # vocab from train only: no leakage from val/test
print(
    f"{len(names)} names -> {len(train_names)} train / {len(val_names)} val"
    f" / {len(test_names)} test, vocab of {len(vocab)} chars"
)

In [ ]:
block_size = 3


def contexts(name_list: list[str]) -> tuple[torch.Tensor, torch.Tensor]:
    pairs = make_ngram_pairs(
        name_list, vocab, block_size + 1
    )  # (N, block_size+1), dernière col = cible
    return pairs[:, :-1], pairs[:, -1]


X, Y = contexts(train_names)
X_val, Y_val = contexts(val_names)
X_test, Y_test = contexts(test_names)
print(f"{X.shape[0]} train / {X_val.shape[0]} val / {X_test.shape[0]} test contexts")

for i in range(5):
    print(f"{''.join(vocab.itos[int(j)] for j in X[i])} ---> {vocab.itos[int(Y[i])]}")

## Build MLP

Following structure of famous paper [A Neural Probabilistic Language Model](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)

Initialization is left to PyTorch's defaults (Kaiming-uniform for `nn.Linear`,
N(0,1) for the embedding), only seeded for reproducibility — overwriting them
with raw randn saturates tanh at step 0. A BatchNorm sits between the hidden
linear and tanh (`batch_norm=True` by default).

In [ ]:
emb_dim, h_dim = 2, 100
model = MLP(vocab, block_size, emb_dim=emb_dim, h_dim=h_dim, seed=42)

## Init diagnostic

With the default (seeded) Kaiming init, the initial loss is ~ln(vocab) — a
uniform softmax — and tanh is far from saturation: the histogram of hidden
pre-activations shows no mass piled up at ±1.

In [ ]:
acts = {}


def capture_pre(module, inputs, output):
    acts["pre"] = output.detach()


hook = model.hidden.register_forward_hook(capture_pre)
with torch.no_grad():
    initial_logits = model(X[:1000])
hook.remove()

init_loss = F.cross_entropy(initial_logits, Y[:1000])
print(f"initial loss {init_loss:.3f} (ln(vocab) = {math.log(len(vocab)):.3f})")
sat = (acts["pre"].abs() > 2).float().mean()
print(f"saturated pre-activations (|z|>2): {sat:.4f}")

fig = go.Figure(go.Histogram(x=acts["pre"].flatten().tolist(), nbinsx=50))
fig.update_layout(xaxis_title="hidden pre-activation", yaxis_title="count")
fig.show()

## Find a maximum learning rate

In [ ]:
# Pick lr (full-batch, can take a while)
lrs = (10 ** torch.linspace(-3, 2, 100)).tolist()
losses = lr_range_test(
    lambda s: MLP(vocab, block_size, emb_dim, h_dim, seed=s), X, Y, lrs, batch_size=64
)
best_lr = plot_lr_range(lrs, losses) / 10
print(f"Maximum learning rate found is {best_lr}")

## Train on train, watch val

In [ ]:
n_steps = 100000
batch_size = 32

model = MLP(vocab, block_size, emb_dim, h_dim, seed=42)
opt = torch.optim.AdamW(model.parameters(), lr=best_lr)  # lr = best_lr du range test
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=best_lr, total_steps=n_steps)
history = train(
    model,
    X,
    Y,
    opt,
    steps=n_steps,
    batch_size=batch_size,
    scheduler=sched,
    seed=42,
    x_val=X_val,
    y_val=Y_val,
    eval_every=100,
    clip_grad=1.0,
)

plot_history(history, smooth=20)
print(
    f"Final train loss {history.train_losses[-1]:.4f}"
    f" | val NLL {history.val[-1][1]:.4f}"
)

## Same run without BatchNorm

Same seed, data and lr with `batch_norm=False`, to see what BN buys on the
validation curve.

In [ ]:
model_nb = MLP(vocab, block_size, emb_dim, h_dim, seed=42, batch_norm=False)
opt_nb = torch.optim.AdamW(model_nb.parameters(), lr=best_lr)
sched_nb = torch.optim.lr_scheduler.OneCycleLR(
    opt_nb, max_lr=best_lr, total_steps=n_steps
)
history_nb = train(
    model_nb,
    X,
    Y,
    opt_nb,
    steps=n_steps,
    batch_size=batch_size,
    scheduler=sched_nb,
    seed=42,
    x_val=X_val,
    y_val=Y_val,
    eval_every=100,
    clip_grad=1.0,
)

plot_history(history_nb)
print(
    f"no BN: final train loss {history_nb.train_losses[-1]:.4f}"
    f" | val NLL {history_nb.val[-1][1]:.4f}"
)

## Sample from the model (min 3 chars)

In [ ]:
model.sample(20)

In [ ]:
model_nb.sample(20)

## Final evaluation on test

Evaluated once, after training on train and hyperparameter choices on val.

In [ ]:
print(
    f"MLP: train NLL {model.nll(X, Y):.4f}"
    f" | val NLL {model.nll(X_val, Y_val):.4f}"
    f" | test NLL {model.nll(X_test, Y_test):.4f}"
)

In [ ]:
print(
    f"MLP: train NLL {model_nb.nll(X, Y):.4f}"
    f" | val NLL {model_nb.nll(X_val, Y_val):.4f}"
    f" | test NLL {model_nb.nll(X_test, Y_test):.4f}"
)